In [58]:
!git pull

There is no tracking information for the current branch.
Please specify which branch you want to merge with.
See git-pull(1) for details.

    git pull <remote> <branch>

If you wish to set tracking information for this branch you can do so with:

    git branch --set-upstream-to=origin/<branch> main



In [60]:
!git remote set-url origin https://github.com/sparkyyyhd-bu/bu-rise-music.git

In [61]:
!git fetch origin
!git checkout audio-based

remote: Enumerating objects: 947, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (183/183), done.
remote: Total 947 (delta 199), reused 149 (delta 85), pack-reused 678 (from 1)
Receiving objects: 100% (947/947), 59.21 MiB | 4.63 MiB/s, done.
Resolving deltas: 100% (623/623), done.
From https://github.com/sparkyyyhd-bu/bu-rise-music
 * [new branch]      audio-based        -> origin/audio-based
 * [new branch]      main               -> origin/main
 * [new branch]      playlist-gen-proto -> origin/playlist-gen-proto
branch 'audio-based' set up to track 'origin/audio-based'.
Switched to a new branch 'audio-based'


In [1]:
!git checkout main
!git pull origin main

error: Your local changes to the following files would be overwritten by checkout:
	.DS_Store
Please commit your changes or stash them before you switch branches.
Aborting
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 93 (delta 69), reused 68 (delta 46), pack-reused 0 (from 0)
Unpacking objects: 100% (93/93), 1.13 MiB | 2.42 MiB/s, done.
From https://github.com/sparkyyyhd-bu/bu-rise-music
 * branch            main       -> FETCH_HEAD
   d3bc146..6eabe13  main       -> origin/main
Updating 10a8adf..6eabe13
Fast-forward
 logs/test_NN_models_fixed.6944064.qlog             |   88 +
 logs/train_CNN_LSTM_fixed.6925286.qlog             |  146 ++
 logs/train_CNN_fixed.6925285.qlog                  |  116 ++
 logs/train_LSTM_fixed.6925287.qlog                 |  201 ++
 logs/train_ResNet_fixed.6929601.qlog               |  108 ++
 logs/train_ViT_fixed.6929599.qlog                  |   94 +
 

In [62]:
from pathlib import Path

# Verify data_utils.py is now present
data_utils_path = Path("src/training/data_utils.py")
print("File exists:", data_utils_path.exists())

File exists: False


In [59]:
import sys
from pathlib import Path

# 1. Search for the file starting from your current notebook location
repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "training" / "data_utils.py").is_file():
    repo_root = repo_root.parent
    if str(repo_root) == repo_root.anchor:
        raise FileNotFoundError("Could not find data_utils.py! Ensure you pulled the audio-based branch.")

# 2. Add it to Python's path and import
sys.path.insert(0, str(repo_root / "src"))
from training.data_utils import grouped_track_id_split

print("Successfully found and imported data_utils.py from:", repo_root)

Successfully found and imported data_utils.py from: /Users/eric/Downloads


In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score



In [10]:
from sklearn.model_selection import GridSearchCV

In [11]:
from sklearn.ensemble import GradientBoostingRegressor

In [12]:
from sklearn.linear_model import LinearRegression

In [13]:
from sklearn.neural_network import MLPRegressor

In [14]:
from sklearn.svm import SVR

In [15]:
from catboost import CatBoostRegressor

In [16]:
from sklearn.model_selection import RandomizedSearchCV

In [17]:
import re

In [39]:
df = pd.read_csv("spotify_tracks.csv")
track_ids = df["id"].astype(str).tolist()

In [40]:
df1 = pd.read_csv("spotify_tracks.csv")
df2 = pd.read_csv("lyrics_features.csv")

In [41]:
df2_clean = df2[df2['mean_syllables_word'] != -1].copy()

# 2. Extract 'id', 'popularity', AND 'lyrics' from df1
df1_extracted = df1[['id', 'popularity', 'lyrics']].copy()

# 3. Clean the 'lyrics' column in df1_extracted before merging
def clean_lyric_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""
    # Remove carriage returns (\r) and newlines (\n) with a space
    text = re.sub(r'[\r\n]+', ' ', text)
    # Collapse multiple spaces into a single space
    text = re.sub(r'\s+', ' ', text)
    # Strip leading and trailing whitespace
    return text.strip()

df1_extracted['lyrics'] = df1_extracted['lyrics'].apply(clean_lyric_text)

# 4. Merge df2_clean with df1_extracted on matching track IDs
df = pd.merge(
    df2_clean, 
    df1_extracted, 
    left_on='track_id', 
    right_on='id', 
    how='inner'
)

# 5. Drop redundant 'id' and 'Unnamed' index columns
df = df.drop(columns=['id'])
columns_to_drop = [col for col in df.columns if 'Unnamed' in col]
df = df.drop(columns=columns_to_drop)

# Verify the lyrics are clean and present
print(df[['track_id', 'popularity', 'lyrics']].head())

                 track_id  popularity  \
0  13keyz9ikBe6ZpRasw7l4X        52.0   
1  1WugzepXsLjnsM0K4UaWYc        55.0   
2  2MO6oEAlMKcsfI8xP3yoy8        46.0   
3  1i4St7fmSUE9nB3R9n8fol        36.0   
4  3UyfvY3Gs6d4wvq8O4ANqQ         6.0   

                                              lyrics  
0  It was one of those times what a real good tim...  
1  Ain't' a lifestyle that I would rather Look in...  
2  Tú, tú eres lo que yo escogí Yo quiero perderm...  
3  Now I've had the time of my life No, I never f...  
4  On est pas là pour payer les dettes On a tous ...  


In [44]:
df['word_density'] = df['n_words'] / (df['n_sentences'] + 1e-5)

In [45]:
from langdetect import detect, DetectorFactory
import numpy as np

DetectorFactory.seed = 0

def get_lang(text):
    try:
        if len(str(text).strip()) < 10:
            return 'unknown'
        return detect(text)
    except:
        return 'unknown'

# 1. Detect language
df['lang'] = df['lyrics'].apply(get_lang)

# 2. Binary indicator for English
df['is_english'] = (df['lang'] == 'en').astype(int)


In [46]:


# 1. Type-Token Ratio (Vocabulary Diversity)
def get_ttr(text):
    words = str(text).lower().split()
    return len(set(words)) / len(words) if len(words) > 0 else 0

# 2. Repetitiveness Score (Popular tracks repeat hooks often!)
def get_repetition(text):
    words = str(text).lower().split()
    if len(words) == 0:
        return 0
    return 1.0 - (len(set(words)) / len(words))

# 3. Average Word Length (Character count per word)
def get_avg_word_length(text):
    words = str(text).split()
    if not words:
        return 0
    return sum(len(w) for w in words) / len(words)

# Apply to DataFrame
df['ttr'] = df['lyrics'].apply(get_ttr)
df['repetition_score'] = df['lyrics'].apply(get_repetition)
df['avg_word_length'] = df['lyrics'].apply(get_avg_word_length)

In [47]:
def extract_advanced_lyric_features(text):
    lines = [line.strip() for line in str(text).split('\n') if line.strip()]
    if not lines:
        return {
            'syllables_per_line': 0,
            'syllables_per_word': 0,
            'syllable_variation': 0,
            'novel_word_prop': 0
        }
    
    # Simple heuristic syllable counter (vowel group count per word)
    def count_syllables(word):
        word = word.lower()
        count = len(re.findall(r'[aeiouy]+', word))
        return max(1, count)

    line_syllables = []
    total_words = 0
    total_syllables = 0
    novel_props = []

    for i, line in enumerate(lines):
        words = re.findall(r'\b\w+\b', line.lower())
        n_words = len(words)
        n_syls = sum(count_syllables(w) for w in words) if words else 0
        
        line_syllables.append(n_syls)
        total_words += n_words
        total_syllables += n_syls

        # Novel Word Proportion between line pair (i-1) and line (i)
        if i > 0:
            prev_words = set(re.findall(r'\b\w+\b', lines[i-1].lower()))
            if words:
                new_words = [w for w in words if w not in prev_words]
                novel_props.append(len(new_words) / len(words))

    avg_syl_line = np.mean(line_syllables) if line_syllables else 0
    avg_syl_word = (total_syllables / total_words) if total_words > 0 else 0
    syl_var = np.std(line_syllables) if len(line_syllables) > 1 else 0
    avg_novel_prop = np.mean(novel_props) if novel_props else 0

    return {
        'syllables_per_line': avg_syl_line,
        'syllables_per_word': avg_syl_word,
        'syllable_variation': syl_var,
        'novel_word_prop': avg_novel_prop
    }

# Apply fast extractor across full dataset
features_df = df['lyrics'].apply(extract_advanced_lyric_features).apply(pd.Series)
df = pd.concat([df, features_df], axis=1)

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Convert lyrics into top word patterns
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=5)
X_tfidf = tfidf.fit_transform(df['lyrics'])

# 2. Compress down to 15 key semantic components
svd = TruncatedSVD(n_components=15, random_state=10)
X_svd = svd.fit_transform(X_tfidf)

# 3. Create DataFrame for SVD features
svd_cols = [f'svd_topic_{i}' for i in range(15)]
df_svd = pd.DataFrame(X_svd, columns=svd_cols, index=df.index)

In [49]:
from textblob import TextBlob

df['sentiment_polarity'] = df['lyrics'].apply(lambda x: TextBlob(str(x)).sentiment.polarity)
df['sentiment_polarity'] = np.where(df['is_english'] == 1, df['sentiment_polarity'], 0)

In [50]:
df.head()

,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,track_id,vocabulary_wealth,popularity,lyrics,word_density,lang,is_english,ttr,repetition_score,avg_word_length,syllables_per_line,syllables_per_word,syllable_variation,novel_word_prop,sentiment_polarity
0,1.10,5.65,31,326,0.043011,13keyz9ikBe6ZpRasw7l4X,0.45,52.0,It was one of those times what a real good tim...,10.516126,en,1,0.296875,0.703125,4.134375,440.0,1.349693,0.0,0.0,0.145833
1,1.37,4.77,74,532,0.050352,1WugzepXsLjnsM0K4UaWYc,0.59,55.0,Ain't' a lifestyle that I would rather Look in...,7.189188,en,1,0.423828,0.576172,4.224609,738.0,1.387218,0.0,0.0,0.057386
2,1.95,3.38,72,430,0.028560,2MO6oEAlMKcsfI8xP3yoy8,0.49,46.0,"Tú, tú eres lo que yo escogí Yo quiero perderm...",5.972221,es,0,0.304651,0.695349,3.730233,688.0,1.600000,0.0,0.0,0.000000
3,1.16,2.99,68,368,0.047849,1i4St7fmSUE9nB3R9n8fol,0.47,36.0,"Now I've had the time of my life No, I never f...",5.411764,en,1,0.335277,0.664723,3.565598,463.0,1.258152,0.0,0.0,0.029545
4,1.32,4.21,39,256,0.040486,3UyfvY3Gs6d4wvq8O4ANqQ,0.60,6.0,On est pas là pour payer les dettes On a tous ...,6.564101,fr,0,0.459574,0.540426,4.127660,353.0,1.378906,0.0,0.0,0.000000


In [78]:
track_ids = df["track_id"].astype(str).tolist()

# 2. Pass your local CSV file explicitly to Lucas's function
train_ids, val_ids, test_ids = map(
    set, 
    grouped_track_id_split(track_ids, metadata_csv="spotify_tracks.csv")
)

# 3. Filter your lyrics DataFrame into leak-free sets
train_df = df[df["track_id"].astype(str).isin(train_ids)].reset_index(drop=True)
val_df   = df[df["track_id"].astype(str).isin(val_ids)].reset_index(drop=True)
test_df  = df[df["track_id"].astype(str).isin(test_ids)].reset_index(drop=True)

print(f"Train tracks: {len(train_df)}")
print(f"Val tracks:   {len(val_df)}")
print(f"Test tracks:  {len(test_df)}")

fixed artist/album-isolated split train=55594 (70.0%), validation=11913 (15.0%), test=11913 (15.0%), fingerprint=27c148725c44eff8
Train tracks: 55594
Val tracks:   11913
Test tracks:  11913


In [60]:
clean_features = [
    'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth',
    'word_density',
    'is_english',
    'repetition_score',
    'avg_word_length',    
    'syllables_per_line',
    'syllables_per_word'
]

X_clean = df[clean_features]
y = df['popularity']

# 2. Get leak-free Grouped Track IDs (70% Train / 15% Val / 15% Test)
track_ids = df['track_id'].astype(str).tolist()
train_ids, val_ids, test_ids = map(
    set, 
    grouped_track_id_split(track_ids, metadata_csv="spotify_tracks.csv")
)

# 3. Create Train / Val / Test Masks
train_mask = df['track_id'].astype(str).isin(train_ids)
val_mask   = df['track_id'].astype(str).isin(val_ids)
test_mask  = df['track_id'].astype(str).isin(test_ids)

X_train, y_train = X_clean[train_mask], y[train_mask]
X_val,   y_val   = X_clean[val_mask],   y[val_mask]
X_test,  y_test  = X_clean[test_mask],  y[test_mask]

# 4. Scale features (Fit ONLY on X_train to prevent leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# 5. Initialize and fit Random Forest model
rf_clean = RandomForestRegressor(
    n_estimators=225, 
    random_state=10, 
    min_samples_split=5, 
    min_samples_leaf=2, 
    max_features='sqrt', 
    max_depth=None,
    n_jobs=-1
)

rf_clean.fit(X_train_scaled, y_train)

# 6. Predict on all splits
train_preds = rf_clean.predict(X_train_scaled)
print(f"Train R² Score : {r2_score(y_train, train_preds):.4f}")

fixed artist/album-isolated split train=55594 (70.0%), validation=11913 (15.0%), test=11913 (15.0%), fingerprint=27c148725c44eff8
Train R² Score : 0.7557


In [62]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import r2_score

# 1. Combine feature matrices cleanly as NumPy arrays
X_train_val_scaled = np.vstack((X_train_scaled, X_val_scaled))

# Combine target Series using values to strip original Pandas indices
y_train_val = np.concatenate((
    y_train.values if isinstance(y_train, (pd.Series, pd.DataFrame)) else y_train,
    y_val.values if isinstance(y_val, (pd.Series, pd.DataFrame)) else y_val
))

# 2. Build test_fold array explicitly matching the exact length of X_train_val_scaled
n_train = len(X_train_scaled)
n_val = len(X_val_scaled)

test_fold = np.array([-1] * n_train + [0] * n_val)

# Verify boundaries before running
assert len(test_fold) == len(X_train_val_scaled) == len(y_train_val), "Lengths must match exactly!"

ps = PredefinedSplit(test_fold=test_fold)

# 3. Setup aggressive parameter space
param_distributions = {
    'n_estimators': [300, 500, 800],
    'max_depth': [None, 35, 50],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2],
    'max_features': [None, 'sqrt', 0.8],
    'bootstrap': [True, False]
}

rf_aggressive = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search_max = RandomizedSearchCV(
    estimator=rf_aggressive,
    param_distributions=param_distributions,
    n_iter=25,
    cv=ps,
    scoring='r2',
    random_state=42,
    verbose=2,
    n_jobs=-1,
    refit=False
)

# Run search
random_search_max.fit(X_train_val_scaled, y_train_val)

# 4. Extract best parameters and fit strictly on Train set
best_params = random_search_max.best_params_
print("\n=== Best Hyperparameters Found ===")
print(best_params)

max_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
max_model.fit(X_train_scaled, y_train)

# Predictions
train_preds = max_model.predict(X_train_scaled)
val_preds   = max_model.predict(X_val_scaled)
test_preds  = max_model.predict(X_test_scaled)

print("\n=== Max R² Performance ===")
print(f"Train R² : {r2_score(y_train, train_preds):.4f}")
print(f"Val R²   : {r2_score(y_val, val_preds):.4f}")
print(f"Test R²  : {r2_score(y_test, test_preds):.4f}")

Fitting 1 folds for each of 25 candidates, totalling 25 fits
[CV] END bootstrap=True, max_depth=50, max_features=sqrt, min_samples_leaf=2, min_samples_split=2, n_estimators=800; total time= 2.1min
[CV] END bootstrap=False, max_depth=50, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=800; total time= 3.0min


/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END bootstrap=True, max_depth=50, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=300; total time=  47.8s
[CV] END bootstrap=False, max_depth=35, max_features=None, min_samples_leaf=2, min_samples_split=2, n_estimators=500; total time= 6.5min
[CV] END bootstrap=True, max_depth=50, max_features=None, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 2.9min
[CV] END bootstrap=False, max_depth=50, max_features=0.8, min_samples_leaf=1, min_samples_split=5, n_estimators=500; total time= 4.7min
[CV] END bootstrap=True, max_depth=50, max_features=0.8, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 2.2min
[CV] END bootstrap=True, max_depth=50, max_features=None, min_samples_leaf=1, min_samples_split=5, n_estimators=800; total time= 6.0min
[CV] END bootstrap=True, max_depth=None, max_features=None, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time= 2.8min
[CV] END bootstrap=True, max_depth=None, max_f

grid search random search


=== Random Forest Performance ===      
'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth'
RMSE (Root Mean Squared Error): 13.2630
MAE  (Mean Absolute Error):    9.9282
R² Score:                       0.4111


r2: 0.4153 250 n-estimators + word_density


In [ ]:
sns.set_theme(style="whitegrid")
plt.figure(figsize=(10, 6))

# Scatter plot with regression trendline (non-linear scatter distribution)
sns.regplot(
    data=clean_df, 
    x='n_words', 
    y='popularity',
    scatter_kws={'alpha': 0.4, 'color': '#1DB954'},  # Spotify Green with opacity
    line_kws={'color': 'darkred', 'linewidth': 2}      # Trendline
)

# Set labels and title
plt.title('Song Word Count (n_words) vs. Spotify Popularity Score', fontsize=14, pad=15)
plt.xlabel('Number of Words in Song (n_words)', fontsize=12)
plt.ylabel('Spotify Popularity Score (0-100)', fontsize=12)
plt.xlim(0, clean_df['n_words'].quantile(0.99))  # Cuts off extreme outliers for a cleaner view

plt.tight_layout()
plt.show()

In [5]:
from pathlib import Path
import sys

In [6]:
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src" / "training" / "data_utils.py").is_file()
)
sys.path.insert(0, str(repo_root / "src"))

# 2. Import the legacy split helper
from training.data_utils import canonical_legacy_track_id_split

In [31]:
clean_features = [
    'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth',
    'word_density',
    'is_english',
    'repetition_score',
    'avg_word_length',    
    'syllables_per_line',
    'syllables_per_word'
]

X_clean = df[clean_features]
y = df['popularity']
track_ids = df['track_id'].astype(str)

# 4. Get the Legacy Reproducible Random Track ID Splits (70 / 15 / 15)
train_ids, val_ids, test_ids = map(
    set, 
    canonical_legacy_track_id_split(metadata_csv="spotify_tracks.csv")
)

# 5. Create Train / Val / Test Masks
train_mask = track_ids.isin(train_ids)
val_mask   = track_ids.isin(val_ids)
test_mask  = track_ids.isin(test_ids)

X_train, y_train = X_clean[train_mask], y[train_mask]
X_val,   y_val   = X_clean[val_mask],   y[val_mask]
X_test,  y_test  = X_clean[test_mask],  y[test_mask]

# 6. Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# 7. Fit model
rf_legacy = RandomForestRegressor(
    n_estimators=225, 
    random_state=10, 
    min_samples_split=5, 
    min_samples_leaf=2, 
    max_features='sqrt', 
    n_jobs=-1
)
rf_legacy.fit(X_train_scaled, y_train)

# 8. Evaluate predictions
train_preds = rf_legacy.predict(X_train_scaled)
val_preds   = rf_legacy.predict(X_val_scaled)
test_preds  = rf_legacy.predict(X_test_scaled)

print("=== Legacy Split Results ===")
print(f"Train R² : {r2_score(y_train, train_preds):.4f}")
print(f"Val R²   : {r2_score(y_val, val_preds):.4f}")
print(f"Test R²  : {r2_score(y_test, test_preds):.4f}")

legacy deterministic random split train=71359, validation=15290, test=15290
=== Legacy Split Results ===
Train R² : 0.7305
Val R²   : 0.4047
Test R²  : 0.4231


tuning hyperparameters

In [33]:
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit

In [55]:
clean_features = [
    'mean_syllables_word',
    'mean_words_sentence',
    'n_sentences',
    'n_words',
    'sentence_similarity',
    'vocabulary_wealth',
    'word_density',
    'is_english',
    'repetition_score',
    'avg_word_length',    
    'syllables_per_line',
    'syllables_per_word',
    'sentiment_polarity'
]

X_clean = df[clean_features]
y = df['popularity']
track_ids = df['track_id'].astype(str)

# 2. Get Legacy Track ID splits
train_ids, val_ids, test_ids = map(
    set, 
    canonical_legacy_track_id_split(metadata_csv="spotify_tracks.csv")
)

train_mask = track_ids.isin(train_ids)
val_mask   = track_ids.isin(val_ids)
test_mask  = track_ids.isin(test_ids)

# 3. Combine Train and Validation sets for Scikit-Learn search tools
X_train_val = pd.concat([X_clean[train_mask], X_clean[val_mask]])
y_train_val = pd.concat([y[train_mask], y[val_mask]])

# Scale combined dataset using train parameters
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_clean[train_mask])
X_val_scaled   = scaler.transform(X_clean[val_mask])
X_test_scaled  = scaler.transform(X_clean[test_mask])

X_train_val_scaled = np.vstack((X_train_scaled, X_val_scaled))

# Create a custom PredefinedSplit (-1 for Train, 0 for Validation)
# This forces RandomizedSearchCV to train ONLY on Train and evaluate ONLY on Validation
split_fold = [-1] * len(X_train_scaled) + [0] * len(X_val_scaled)
ps = PredefinedSplit(test_fold=split_fold)

# 4. Comprehensive Hyperparameter Space
param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.5, 0.8, None],
    'bootstrap': [True, False]
}

# 5. Initialize RandomizedSearchCV (Testing 30 random combinations across the grid)
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_distributions,
    n_iter=30,
    cv=ps,
    scoring='r2',
    random_state=42,
    verbose=2,
    n_jobs=-1
)

# Run search
random_search.fit(X_train_val_scaled, y_train_val)

# 6. Extract Best Model and Evaluate
best_model = random_search.best_estimator_

train_preds = best_model.predict(X_train_scaled)
val_preds   = best_model.predict(X_val_scaled)
test_preds  = best_model.predict(X_test_scaled)

print("\n=== Legacy Randomized Search Complete ===")
print("Best Parameters:", random_search.best_params_)
print(f"Train R² : {r2_score(y[train_mask], train_preds):.4f}")
print(f"Val R²   : {r2_score(y[val_mask], val_preds):.4f}")
print(f"Test R²  : {r2_score(y[test_mask], test_preds):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y[test_mask], test_preds)):.4f}")

legacy deterministic random split train=71359, validation=15290, test=15290
Fitting 1 folds for each of 30 candidates, totalling 30 fits
[CV] END bootstrap=True, max_depth=10, max_features=0.5, min_samples_leaf=2, min_samples_split=2, n_estimators=300; total time=  41.7s
[CV] END bootstrap=True, max_depth=None, max_features=0.5, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 1.4min


/opt/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END bootstrap=True, max_depth=30, max_features=sqrt, min_samples_leaf=1, min_samples_split=15, n_estimators=300; total time=  40.7s
[CV] END bootstrap=False, max_depth=30, max_features=sqrt, min_samples_leaf=4, min_samples_split=5, n_estimators=500; total time= 1.6min
[CV] END bootstrap=False, max_depth=15, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=  14.8s
[CV] END bootstrap=True, max_depth=None, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=200; total time=  29.3s
[CV] END bootstrap=True, max_depth=15, max_features=0.8, min_samples_leaf=2, min_samples_split=15, n_estimators=100; total time=  33.8s
[CV] END bootstrap=False, max_depth=30, max_features=None, min_samples_leaf=4, min_samples_split=2, n_estimators=500; total time= 3.6min

=== Legacy Randomized Search Complete ===
Best Parameters: {'n_estimators': 300, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30, 'bootst

In [35]:
df.shape

(79420, 20)

In [37]:
df

,mean_syllables_word,mean_words_sentence,n_sentences,n_words,sentence_similarity,track_id,vocabulary_wealth,popularity,lyrics,word_density,lang,is_english,sentiment_polarity,ttr,repetition_score,avg_word_length,syllables_per_line,syllables_per_word,syllable_variation,novel_word_prop
0,1.10,5.65,31,326,0.043011,13keyz9ikBe6ZpRasw7l4X,0.45,52.0,It was one of those times what a real good tim...,10.516126,en,1,0.145833,0.296875,0.703125,4.134375,440.0,1.349693,0.0,0.0
1,1.37,4.77,74,532,0.050352,1WugzepXsLjnsM0K4UaWYc,0.59,55.0,Ain't' a lifestyle that I would rather Look in...,7.189188,en,1,0.057386,0.423828,0.576172,4.224609,738.0,1.387218,0.0,0.0
2,1.95,3.38,72,430,0.028560,2MO6oEAlMKcsfI8xP3yoy8,0.49,46.0,"Tú, tú eres lo que yo escogí Yo quiero perderm...",5.972221,es,0,0.000000,0.304651,0.695349,3.730233,688.0,1.600000,0.0,0.0
3,1.16,2.99,68,368,0.047849,1i4St7fmSUE9nB3R9n8fol,0.47,36.0,"Now I've had the time of my life No, I never f...",5.411764,en,1,0.029545,0.335277,0.664723,3.565598,463.0,1.258152,0.0,0.0
4,1.32,4.21,39,256,0.040486,3UyfvY3Gs6d4wvq8O4ANqQ,0.60,6.0,On est pas là pour payer les dettes On a tous ...,6.564101,fr,0,0.000000,0.459574,0.540426,4.127660,353.0,1.378906,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79415,1.24,4.00,72,501,0.022300,5gPL7XvlYmX922oyxscYH4,0.60,54.0,"Now, should you expect to see something that y...",6.958332,en,1,0.021726,0.394251,0.605749,3.905544,642.0,1.281437,0.0,0.0
79416,1.12,4.52,31,272,0.152688,64iZLIYuqRR8rNedN0Yvnh,0.40,42.0,(S. Day & are. St. Clare) Well I've been looki...,8.774191,en,1,0.343906,0.330544,0.669456,3.535565,314.0,1.154412,0.0,0.0
79417,1.30,3.90,39,257,0.021592,5bylaCp22328sFwlzLNC0n,0.64,42.0,Had a scratch only you could itch Underneath t...,6.589742,en,1,0.035833,0.447154,0.552846,3.987805,367.0,1.428016,0.0,0.0
79418,1.48,3.95,41,241,0.018293,7DVr6Az7HukAZeK7v3cGZx,0.60,38.0,Get on the A train Get on the right track If y...,5.878047,en,1,0.235786,0.414530,0.585470,3.918803,336.0,1.394191,0.0,0.0


In [38]:
print("Total rows in df:", len(df))
print("Train rows in df:", train_mask.sum())
print("Val rows in df:  ", val_mask.sum())
print("Test rows in df: ", test_mask.sum())

Total rows in df: 79420
Train rows in df: 55465
Val rows in df:   11979
Test rows in df:  11976
